# Implementing a simple AI Agent

In this lab, we will implement a simple AI agent that is able to retrive information from a datasource. This will be done by a mechanic called "Tool calling". 

Let us demystify the concept of an AI Agent. An AI agent is nothig more (**or less**) than a software program build around an LLM that allows the LLM to take autonomous decisions and use tools. Let us break this down:
- A "tool" is an object that allow LLMs to interact with external environments. These tools may be functions made available to LLMs such as a calculator, a search engine, or even another AI—to perform specific tasks it isn't designed to handle on its own. The tools can be executed separately whenever the LLM determines that their use is appropriate.
- A LLM trained to perform supports tool calling (more on this later). Basically, this means that the LLM is able to decide when to use a "tool" .
- Tool calling enables the model to generate a response for a prompt that aligns with a user-defined schema for a function.


that is able to interact with its environment. This interaction can be done in many ways, such as:

## Installation

The first choice to be made is which LLM model to use. Start with small models. The code below has been tested with `llama3.1:8b`  (note: **llama3.1:8b is about 5GB**). Other models may require some adjustments to the prompt. Check the documentation of the model you want to use or test. For instance, if you want to use 'llama3.2:1b' (a smaller model), check the doc in the "Tool Calling (1B/3B)" section, in the [official web site](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_2/).

In [1]:
model_name = 'llama3.1:8b'

### On Kaggle
Repeat the instructions from the previous part of the lab to install and run Ollama on Kaggle.
The last instruction should print: 

```python
print("Ollama server is running on 127.0.0.1:11438")
```

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', model_name])

### On your local machine
Be sure that Ollama is runnung (follow the instructions in the prrevious lab). In the terminal, pull the `llama3.1:8b` model or another [tool model](https://ollama.com/search?c=tools). Start with small models. The code below has been tested with `llama3.1:8b`  (**llama3.1:8b is about 5GB**). Other models may require some adjustments in the prompt. Check the documentation of the model you want to use or test other models.

```bash
ollama run llama3.1:8b
```

---

## Teach your model how tp interact with a home automation system (example)


### Step 0 [optional]: get the prompt using speech-to-text

In [15]:
from ipywebrtc import AudioRecorder, CameraStream
from IPython.display import Audio
import ffmpeg

NOTE: I could not manage to make the next ipywebrtc widget to work on VSCODE or PyCharm. It worked in JupyterLab.
If you used poetry as a package manager, you can launch jupyterLab using the command below in a terminal. 

```bash
poetry run jupyter lab 
```

The idea is to run until the next two cells in JupyterLab. The first cell will use the microphone to record your voice. The second cell saves the audio file in the current directory.
After that, everything can be run in VSCODE or PyCharm.

In [16]:
# Create a camera stream
camera = CameraStream(constraints={'audio': True, 'video': False})
recorder = AudioRecorder(stream=camera)
recorder

AudioRecorder(audio=Audio(value=b'', format='webm'), stream=CameraStream(constraints={'audio': True, 'video': …

In [ ]:
# Save the recorded audio to a file in the ./audio folder
# I want to give the path in a os independent way, so I will use os.path.join
import os
filename = os.path.join('audio', 'recording.webm')
print(filename)
with open(filename, 'wb') as f:
    f.write(recorder.audio.value)



In [ ]:
import subprocess

try:
    subprocess.run(["ffmpeg", "-version"], check=True)
    print("FFmpeg is accessible")
except FileNotFoundError:
    print("FFmpeg is not accessible")

In [ ]:
# Convert the .webm file to .wav format
input_file = filename
output_file = os.path.join('audio', 'recording.wav')

NOTE: to make ffmpeg work, you need to install it (surprise!). Follow the instructions on the web. 
For Windows, you can use [this](https://www.geeksforgeeks.org/how-to-install-ffmpeg-on-windows/) guide. On Mac, you can use [this](https://phoenixnap.com/kb/ffmpeg-mac) guide (*to be tested*).

In [ ]:
# check if the output file already exists
if os.path.exists(output_file):
    print(f"WARNING: Output file {output_file} already exists. Please delete it before running the conversion.")
else:
    # Run the ffmpeg command to convert the file
    ffmpeg.input(input_file).output(output_file).run()
    print(f"Converted {input_file} to {output_file}")


Running the code below, you can replay the audio to check if the conversion was successful. 

In [17]:
from scipy.io import wavfile

# Load the WAV file
sample_rate, data = wavfile.read(output_file)

# Play the audio (for testing)
Audio(data=data, rate=sample_rate)

NameError: name 'output_file' is not defined

Finally, [whisper](https://github.com/openai/whisper) manages the conversion audio to text. The code below will convert the audio to text.
The `prompt` variable will contain the prompt to be used in the next step. 

NOTE: the first time you run the code below, it will take a while to download the model.

In [ ]:
import whisper

# Load the Whisper model
model = whisper.load_model("base")

# Transcribe the audio file
result = model.transcribe(output_file)

# Print the transcribed text
prompt = result["text"]
print(prompt)

### Step 1: Import the necessary libraries

We use `ChatResponse` and `chat`.

In [18]:
from ollama import ChatResponse, chat

### Step 2: Define the functions (tools)

In this example, we define two functions: `add_two_numbers` and `subtract_two_numbers`. These functions will be used by the model to perform simple calculations.

Basically, each function defines a prompt for the model and what the model should do with the input. As you can see in the example, the prompte structure is does not need to be complex or always the same.

In [3]:
# when not using the docker on the local machine, the url is different
# students will need to use: https://homeio.sdi.hevs.ch/homeio (not sure about the port)

In [4]:
is_localhost = False

if is_localhost:
    url = "http://localhost:8080/homeio"
else:
    url = "https://homeio.sdi.hevs.ch/homeio"

In [5]:
# We strictly define the types of the arguments and the return value to be sure that the tool will work as expected
import requests
import config

def open_garage_door() -> str:
  """
  Open the garage door

  Args:
    The function does not take any arguments

  Returns:
    The text of the response from the request
  """
  # URL
  global url

  # Parameters from the URL query string
  params = {'room': 'Garage', 'device': 'Door'}

  # Headers
  headers = {
      'accept': 'application/json',
      'Content-Type': 'application/json'
  }

  # Data (request body)
  data = {
      "command": "UP"
  }

  # Replace with your actual username and password
  username = config.username
  password = config.password

  try:
      response = requests.post(
          url,
          params=params,
          headers=headers,
          json=data,  # Use 'json' parameter to automatically encode the data as JSON
          auth=(username, password)  # For basic authentication
      )

      # Print the response status code
      print(f"Status Code: {response.status_code}")

      # Print the response content (if any)
      try:
          print("Response JSON:", response.json())
      except requests.exceptions.JSONDecodeError:
          print("Response Content:", response.text)

      # You can add more error handling or processing of the response here

  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")

  return response.text


In [ ]:
def close_garage_door() -> str:
  """
  Close the garage door

  Args:
    The function does not take any arguments

  Returns:
    The text of the response from the request
  """
  # URL
  global url

  # Parameters from the URL query string
  params = {'room': 'Garage', 'device': 'Door'}

  # Headers
  headers = {
      'accept': 'application/json',
      'Content-Type': 'application/json'
  }

  # Data (request body)
  data = {
      "command": "DOWN"
  }

  # Replace with your actual username and password
  username = 'sdi42'
  password = '495450f1c928f0828926047ad396d5ac'

  try:
      response = requests.post(
          url,
          params=params,
          headers=headers,
          json=data,  # Use 'json' parameter to automatically encode the data as JSON
          auth=(username, password)  # For basic authentication
      )

      # Print the response status code
      print(f"Status Code: {response.status_code}")

      # Print the response content (if any)
      try:
          print("Response JSON:", response.json())
      except requests.exceptions.JSONDecodeError:
          print("Response Content:", response.text)

      # You can add more error handling or processing of the response here

  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")

  return response.text


In [20]:
def manage_shutters(room: str, device: str, command: str) -> str:
  """
  Manage the opening and closing of the shutters in the home automation system
  Args:
    room (str): The room where the shutters are located (for instance, LivingRoom, SingleBedroom, Laundry.)
    device (str): The name of the shutters (for instance, ShuttersWest, or ShuttersSouth)
    command (str): The command to be executed (UP or DOWN)

  Returns:
    The text of the response from the request
  """
  # URL
  global url

  # Parameters from the URL query string
  params = {'room': room, 'device': device}

  # Headers
  headers = {
      'accept': 'application/json',
      'Content-Type': 'application/json'
  }

  # Data (request body)
  data = {
      "command": command # UP or DOWN
  }

  # Replace with your actual username and password
  username = config.username
  password = config.password

  try:
      response = requests.post(
          url,
          params=params,
          headers=headers,
          json=data,  # Use 'json' parameter to automatically encode the data as JSON
          auth=(username, password)  # For basic authentication
      )

      # Print the response status code
      print(f"Status Code: {response.status_code}")

      # Print the response content (if any)
      try:
          print("Response JSON:", response.json())
      except requests.exceptions.JSONDecodeError:
          print("Response Content:", response.text)

      # You can add more error handling or processing of the response here

  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")

  return response.text


In [21]:
def manage_lights(room: str, device: str, command: str) -> str:
  """
  Turns on or off the lightsin the home automation system
  Args:
    room (str): The room where the light is located (for instance, LivingRoom, SingleBedroom, Laundry, Corridor.)
    device (str): The name of the light (it is always "Light")
    command (str): The command to be executed (ON or OFF)

  Returns:
    The text of the response from the request
  """
  # URL
  global url

  # Parameters from the URL query string
  params = {'room': room, 'device': device}

  # Headers
  headers = {
      'accept': 'application/json',
      'Content-Type': 'application/json'
  }

  # Data (request body)
  data = {
      "command": command # ON or OFF
  }

  # Replace with your actual username and password
  username = config.username
  password = config.password

  try:
      response = requests.post(
          url,
          params=params,
          headers=headers,
          json=data,  # Use 'json' parameter to automatically encode the data as JSON
          auth=(username, password)  # For basic authentication
      )

      # Print the response status code
      print(f"Status Code: {response.status_code}")

      # Print the response content (if any)
      try:
          print("Response JSON:", response.json())
      except requests.exceptions.JSONDecodeError:
          print("Response Content:", response.text)

      # You can add more error handling or processing of the response here

  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")

  return response.text


In [22]:
# We create a dictionary that maps the function names to the functions

available_functions = {
  'open_garage_door': open_garage_door,
  'close_garage_door': close_garage_door,
  'manage_shutters': manage_shutters,
  'manage_lights': manage_lights
}

### Step 3: Define the chat with the models.
We define the message, we inform the model about the available tools, and we call the model.

In [24]:
prompt = "Can you switch off the light in the Garage and close the door?"

In [25]:
messages = [{'role': 'user', 'content': prompt}]
print('Prompt:', messages[0]['content'])

response: ChatResponse = chat( # here the `:` say that response is of type ChatResponse
  model_name, # change this to the model you want to use
  messages=messages,
  tools=[open_garage_door, close_garage_door, manage_shutters, manage_lights], # we indicate the tools available to the model
)

Prompt: Can you switch off the light in the Garage and close the door?


### Step 4: We process the response

The idea is that the model decides whether to use the tool or not. The choice of the model is stored in the variable `response.message.tool_calls`. ToolCall is a list of the tools that the models decided to use.

In [26]:
print('Response:', response.message.tool_calls) # print this for debugging/undersanding the tool calls

Response: [ToolCall(function=Function(name='manage_lights', arguments={'command': 'OFF', 'device': 'Light', 'room': 'Garage'})), ToolCall(function=Function(name='close_garage_door', arguments={'w': 'Close'}))]


**Question**: The response contains the tool(s) to call with its arguments. Try to ask a question that is not related to the tools. What does the model do? 

**Answer**: The response.message.tool_calls list will be empty or propose non-existent tools.

In the code below, we test whether the model decided to use one or several tools.

NOTE: the `**` operator is used to pass a dictionary as keyword arguments. In summary, in the code below, we assign to `function_to_call` the function that the model decided to use (using the `:=` operator) and we pass the arguments to the function (using the `**` operator).

In [27]:
functions_called = ""

# we check if the model returned any tool calls
if response.message.tool_calls:
  # There may be multiple tool calls in the response, we treat them one by one ina for loop
  for tool in response.message.tool_calls:
    # Ensure the function is available, and then call it
    
    function_to_call = available_functions.get(tool.function.name) # here we get the function to call from the dictionary
    if function_to_call:
      print('Calling function:', tool.function.name)
      print('Arguments:', tool.function.arguments)
      # The ** operator unpacks the dictionary into keyword arguments
      # In the example 'What is three minus one?', the instruction below will be equivalent to call 'add_two_numbers(a=3, b=1)'
      output = function_to_call(**tool.function.arguments)
      print('Function output:', output)

      functions_called += f"Function {tool.function.name} called with arguments {tool.function.arguments} and output {output}\n"
    else:
      print('Function', tool.function.name, 'not found')



Calling function: manage_lights
Arguments: {'command': 'OFF', 'device': 'Light', 'room': 'Garage'}
Status Code: 200
Response JSON: {'status': 'ok'}
Function output: {"status":"ok"}
Calling function: close_garage_door
Arguments: {'w': 'Close'}
Status Code: 200
Response JSON: {'status': 'ok'}
Function output: {"status":"ok"}


### Step 5: The final "ribbon"	
We use the LLM once again to process the response and print the result in a nice way.

In [14]:
final_prompt = "You where able to successfully call the function(s):\n" + functions_called + "\n\n" + "Now, please provide a final answer to the user based on the function outputs and the original prompt:\n" + prompt

messages = [{'role': 'user', 'content': final_prompt}]
# Only needed to chat with the model using the tool call results
if response.message.tool_calls:
  # Add the function response to messages for the model to use
  messages.append(response.message)
  messages.append({'role': 'tool', 'content': str(output), 'name': tool.function.name})

else: 
  print('No tool calls returned from model, just providing the response to the user...')
  # we do not need to modify the messages if the model did not return any tool calls

# Get final response from model with function outputs
final_response = chat(model_name, messages=messages)
print('Final response:', final_response.message.content)


Final response: Here is the final answer:

Yes, I was able to successfully switch on the light in the Garage and open the door.
